In [12]:
from langchain_core.documents import Document

In [13]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python is a high-level, interpreted.txt",encoding = "utf-8")

In [14]:
document = loader.load()

In [15]:
document

[Document(metadata={'source': 'data/Python is a high-level, interpreted.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific 

In [16]:
#from langchain_community.document_loaders.pdf import PyPDFLoader

#pdf_loader = PyPDFLoader("data/research2.pdf")
#document = pdf_loader.load()
#document

## Ingestion Pipeline

In [17]:
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents conversation

In [18]:
def load_pdf():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):

            pdf_path = os.path.join(folder_path, filename)
    
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs +=1

    print("total pdfs", num_docs)
    return all_docs

In [19]:
all_pdf_documents = load_pdf()

total pdfs 2


### Chunks

In [20]:
#!pip install langchain_text_splitters

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(documents)
    return chunked_doc

In [22]:
chunks = split_doc(all_pdf_documents)

In [23]:
len(chunks)

320

### Embedding

In [24]:
from sentence_transformers import SentenceTransformer

In [25]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading...",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar = True)
        print("embeddings shape",embeddings.shape)
        return embeddings

In [26]:
embedding_manager = EmbeddingManager()

Loading... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions =  384


C:\Users\darla\AppData\Local\Temp\ipykernel_3132\1380835229.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions = ",self.model.get_sentence_embedding_dimension())


### Vector store

In [27]:
import chromadb
import uuid

In [28]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count()) 

In [29]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 320


In [30]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape (320, 384)
total documents added in vector store= 320
docs in collection: 640


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [33]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [34]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 5 documents


[{'id': 'doc_a17c2525-1b52-4bd5-8b67-41a22ad495da',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'page': 6,
   'producer': 'PyPDF2',
   'lastpage': '6008',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million par

## Ingerate with LLms

In [61]:
# Using GROQ AI
API_Key_GROQ = "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"

In [62]:
#!pip install langchain-groq

In [63]:
from langchain_groq import ChatGroq
llms = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model = "qwen/qwen3-32b",
    temperature = 0.1,
    max_tokens = 1024
)

In [64]:
# generate our retrieval- augmented output

def generate_output(query, rag_retriever,llms,top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"]for doc in results]) if results else ""

    if not context:
        print("we found no relevant content ")
    # context + query
    prompt = f"""use given context to generate the answer 
                Context: {context}
                Query: {query}"""

    response = llms.invoke([prompt.format(context = context,query = query)])
    return response.content

In [65]:
answer = generate_output("what is RAG?",rag_retriever,llms)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 3 documents


In [67]:
print(answer)

<think>
Okay, the user is asking "what is RAG?" and I need to use the provided context to generate the answer. Let me look through the context again.

The context mentions that the survey presents a thorough review of state-of-the-art RAG methods, discussing its evolution through paradigms like naive RAG and an effective RAG framework. It also talks about evaluation methods, tasks, datasets, and future directions. The paper's sections include an introduction to the main concept and current paradigms in Section II.

So, RAG stands for Retrieval-Augmented Generation. From the context, it seems RAG combines retrieval of information with generation to enhance model responses. The paradigms mentioned are naive RAG and effective RAG, which probably refer to different approaches or stages in the development of RAG methods. The survey covers the evolution of these methods, evaluation metrics, and future trends.

I should define RAG as a method that uses retrieval and generation. Mention the pa

In [68]:
answer = generate_output("what is encoder - decoder ?",rag_retriever,llms)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 3 documents


In [71]:
print(answer)

<think>
Okay, the user is asking "what is encoder-decoder?" and I need to use the provided context to answer. Let me start by reading through the context carefully.

The context talks about the encoder and decoder each having 6 identical layers. The encoder's layers have two sub-layers, while the decoder has three. The third sub-layer in the decoder does multi-head attention on the encoder's output. Both use residual connections and layer normalization. The model dimension is 512, and there's mention of positional encodings and a dropout rate of 0.1.

So, the encoder-decoder structure is a common architecture in models like Transformers. The encoder processes input data, and the decoder generates output. The key points from the context are the number of layers, sub-layers, attention mechanisms, residual connections, normalization, and positional encodings.

I should explain that the encoder-decoder consists of multiple layers. The encoder has two sub-layers per layer (maybe self-attent